In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [3]:
spark=SparkSession.builder.appName("Join").getOrCreate()

In [4]:
products=spark.read.csv("Dimproduct.csv",header=True,inferSchema=True)
products=products.select("ProductKey","ProductSubcategoryKey","StandardCost","FinishedGoodsFlag","Color","Size","Weight")
products=products.filter(F.col("productkey").rlike("^[0-9]+$"))


In [5]:
fact_sales=spark.read.csv("FactInternetSales.csv",header=True,inferSchema=True)
sales_products=fact_sales.join(products,fact_sales.ProductKey==products.ProductKey,"left")\
                         .select(fact_sales.SalesOrderNumber,fact_sales.ProductKey,products.StandardCost,products.FinishedGoodsFlag,products.Color,products.Size,products.Weight)      
sales_products.printSchema()

sales_products.select("SalesOrderNumber","ProductKey","StandardCost","FinishedGoodsFlag","Color","Size","Weight").show(5,truncate=False)


root
 |-- SalesOrderNumber: string (nullable = true)
 |-- ProductKey: integer (nullable = true)
 |-- StandardCost: double (nullable = true)
 |-- FinishedGoodsFlag: integer (nullable = true)
 |-- Color: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- Weight: double (nullable = true)

+----------------+----------+------------+-----------------+------+----+------+
|SalesOrderNumber|ProductKey|StandardCost|FinishedGoodsFlag|Color |Size|Weight|
+----------------+----------+------------+-----------------+------+----+------+
|SO43697         |310       |2171.2942   |1                |Red   |62  |15.0  |
|SO43698         |346       |1912.1544   |1                |Silver|44  |21.13 |
|SO43699         |346       |1912.1544   |1                |Silver|44  |21.13 |
|SO43700         |336       |413.1463    |1                |Black |62  |20.0  |
|SO43701         |346       |1912.1544   |1                |Silver|44  |21.13 |
+----------------+----------+------------+----------------

In [6]:
#create a df with recent orders from customer and their product details
customer_window=Window.partitionBy("CustomerKey").orderBy(F.col("OrderDateKey").desc())
recent_orders=fact_sales.withColumn("rownumber",F.row_number().over(customer_window))\
                        .filter(F.col("rownumber")==1).drop("rownumber")
print(fact_sales.count())
print(recent_orders.count())

60398
18484


In [7]:
fact_sales = fact_sales.withColumn("OrderDate", F.to_date(F.col("OrderDate"), "d MMM yy"))

fact_sales = fact_sales.withColumn("order_month", F.month(F.col("OrderDate"))) \
       .withColumn("order_year",  F.year(F.col("OrderDate")))

fact_sales.show(5, truncate=False)


+----------+------------+----------+-----------+-----------+------------+-----------+-----------------+----------------+--------------------+--------------+-------------+---------+--------------+--------------------+--------------+-------------------+----------------+-----------+--------+-------+---------------------+----------------+----------+---------+--------+-----------+----------+
|ProductKey|OrderDateKey|DueDateKey|ShipDateKey|CustomerKey|PromotionKey|CurrencyKey|SalesTerritoryKey|SalesOrderNumber|SalesOrderLineNumber|RevisionNumber|OrderQuantity|UnitPrice|ExtendedAmount|UnitPriceDiscountPct|DiscountAmount|ProductStandardCost|TotalProductCost|SalesAmount|TaxAmt  |Freight|CarrierTrackingNumber|CustomerPONumber|OrderDate |DueDate  |ShipDate|order_month|order_year|
+----------+------------+----------+-----------+-----------+------------+-----------+-----------------+----------------+--------------------+--------------+-------------+---------+--------------+--------------------+----

In [8]:
import traceback

try:
    fact_sales.write \
      .mode("overwrite") \
      .option("compression", "snappy") \
      .partitionBy("order_year", "order_month") \
      .parquet("C:/Users/AshaSurabhi/Project/sales")
except Exception as e:
    print(str(e))

In [9]:
import os
print(os.environ.get("HADOOP_HOME"))

C:\Users\AshaSurabhi\Hadoop
